<a href="https://colab.research.google.com/github/nadianaz0514-cloud/urdu-ocr-codesaviours-si26-Nadia/blob/main/SI26_Week2_Nadia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook preprocesses Urdu text images by converting them to grayscale, resizing them, removing noise, and creating clean binary images. It also tests Tesseract OCR on the processed images to evaluate its performance on Urdu text and identify its limitations for OCR.

In [1]:
!pip install opencv-python-headless pillow

import cv2
import numpy as np
from PIL import Image
import os
import matplotlib.pyplot as plt

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [2]:
def preprocess_image(image_path, save_path):
    img = cv2.imread(image_path)

    if img is None:
        print(f"Could not load: {image_path}")
        return

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (512, 128))
    denoised = cv2.fastNlMeansDenoising(resized, h=10)
    _, binary = cv2.threshold(denoised, 127, 255, cv2.THRESH_BINARY)

    cv2.imwrite(save_path, binary)
    return binary

os.makedirs("data/processed", exist_ok=True)
print("Preprocessing function ready!")

Preprocessing function ready!


In [3]:
import os

folders = [
    'data2/raw/newspaper',
    'data2/raw/books',
    'data2/raw/signboards',
    'data2/raw/synthetic',
    'data2/raw/other'
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f'Created: {folder}')

print('Folder structure ready!')

Created: data2/raw/newspaper
Created: data2/raw/books
Created: data2/raw/signboards
Created: data2/raw/synthetic
Created: data2/raw/other
Folder structure ready!


In [4]:
import glob

all_images = glob.glob("data2/raw/**/*.jpg", recursive=True)
all_images += glob.glob("data2/raw/**/*.png", recursive=True)

print(f"Found {len(all_images)} images to process")

processed_count = 0

for img_path in all_images:
    filename = os.path.basename(img_path)
    save_path = f"data/processed/{filename}"
    result = preprocess_image(img_path, save_path)

    if result is not None:
        processed_count += 1

print(f"Done! Processed {processed_count} images")
print("Check data/processed/ folder")

Found 100 images to process
Done! Processed 100 images
Check data/processed/ folder


In [5]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [7]:
!pip install pytesseract
!apt-get update
!apt-get install -y tesseract-ocr
!apt-get install -y tesseract-ocr-urd

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [101 kB]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.6 MB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,129 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,396 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages

In [9]:
import pytesseract
from PIL import Image

In [19]:
import glob
from PIL import Image
import pytesseract

all_images = glob.glob("/content/data2/raw/**/*.jpg", recursive=True)
all_images += glob.glob("/content/data2/raw/**/*.png", recursive=True)

candidates = []

for img_path in all_images:
    img = Image.open(img_path)
    text = pytesseract.image_to_string(img, lang="urd", config="--oem 3 --psm 6")
    word_count = len(text.split())
    candidates.append((img_path, word_count))

short_text_images = [c for c in candidates if 1 <= c[1] <= 10]
test_images = [c[0] for c in short_text_images[:5]]

print("Selected images:", test_images)
print("=== Tesseract Results on Urdu Images ===\n")

for i, img_path in enumerate(test_images, start=1):
    img = Image.open(img_path)

    result = pytesseract.image_to_string(
        img,
        lang="urd",
        config="--oem 3 --psm 6"
    )

    print(f"Image {i}")
    print("Tesseract Output:")
    print(result)
    print("-" * 50)

Selected images: ['/content/data2/raw/books/textbox_3_line_1_.png', '/content/data2/raw/books/textbox_5_line_3_.png', '/content/data2/raw/other/829.png', '/content/data2/raw/other/841.png', '/content/data2/raw/other/849.png']
=== Tesseract Results on Urdu Images ===

Image 1
Tesseract Output:
ہو ٹل پا سافٰ افر علہ 20 اور حوسانر

--------------------------------------------------
Image 2
Tesseract Output:
- ھا پکل| ہے گرد ٦٭د‏ تھا ۔

--------------------------------------------------
Image 3
Tesseract Output:
سم(

--------------------------------------------------
Image 4
Tesseract Output:
سْ
سس

--------------------------------------------------
Image 5
Tesseract Output:
ڑ2

--------------------------------------------------


Gap Analysis

Image 1 (textbox_3_line_1_.png)
- Actual Urdu Text: ہوٹل کا سٹاف اور عملہ نہایت اچھا اور دوستانہ
- Tesseract Output: "ہو ٹل پا سافٰ افر علہ 20 اور حوسانر"
- What went wrong: The output produced disconnected words that don't form a coherent sentence. Character shapes were roughly recognized, but sequencing and spacing were incorrect, resulting in a meaningless combination.

Image 2 (textbox_5_line_3_.png)
- Actual Urdu Text: کمرے کا پنکھا بہت گرد آلود تھا
- Tesseract Output: "- ھا پکل| ہے گرد ٦٭د‏ تھا ۔"
- What went wrong: Wrong characters were produced, along with a stray number/symbol (٦, |) that likely wasn't in the original text. Segmentation failed, breaking letters apart incorrectly.

Image 3 (829.png)
- Actual Urdu Text: ش
- Tesseract Output: "سم("
- What went wrong: Most of the text was missed entirely — only a tiny fragment was returned, even though the image likely contained a full word or phrase. Poor recognition caused nearly everything to be dropped.

Image 4 (841.png)
- Actual Urdu Text: ش
- Tesseract Output: "سْ\nسس"
- What went wrong: Complete gibberish — only random, disconnected letters appeared with no meaningful word formed. Character-level misrecognition was the main issue.

Image 5 (849.png)
- Actual Urdu Text: ش
- Tesseract Output: "ڑ2"
- What went wrong: Nearly all the text was missed, with only a single letter and a number (likely noise) detected. This shows severe under-recognition.

Summary
Across all 5 short-text images, Tesseract's performance was even weaker than on full paragraphs. This suggests Tesseract struggles more with short, isolated text snippets or single lines, since it lacks the surrounding context that longer paragraphs provide, which normally helps it correct or refine its predictions.